In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("base_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/process/ETL_Clm", "Base Path")
dbutils.widgets.text("file1_prefix", "ODM.EDW.VEN100FA.WEEKLY.ASCII.PROD", "File 1 Prefix (before date)")
dbutils.widgets.text("file2_prefix", "ODM.EDW.VEN100FA.WEEKLY.ASCII.PROD_01", "File 2 Prefix (before date)")
dbutils.widgets.text("column_name", "CLM_TYP_CD", "Column to Validate")
dbutils.widgets.text("error_report_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_clm", "Error Report Table")

In [0]:
BASE = dbutils.widgets.get("base_path").rstrip("/")
FILE1_PREFIX = dbutils.widgets.get("file1_prefix")
FILE2_PREFIX = dbutils.widgets.get("file2_prefix")
COL_NAME = dbutils.widgets.get("column_name")
ERROR_TABLE = dbutils.widgets.get("error_report_table")

print(f"base_path={BASE}")
print(f"file1_prefix={FILE1_PREFIX}")
print(f"file2_prefix={FILE2_PREFIX}")
print(f"column_name={COL_NAME}")
print(f"error_report_table={ERROR_TABLE}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
import re


nb_start = datetime.now()
nb_start_str = nb_start.strftime("%Y-%m-%d %H:%M:%S")


ALLOWED_V1 = [
    'PART A INSTITUTIONAL','PART B INSTITUTIONAL','PART C INSTITUTIONAL','LTC','INPATIENT',
    'PART B INPATIENT','PART C LTC','PART A OUTPATIENT','INSTITUTIONAL','PART C OUTPATIENT',
    'PART A LTC','PART B OUTPATIENT','PART B LTC','PART A INPATIENT','PART C INPATIENT',
    'OUTPATIENT','PART C PROFESSIONAL','DENTAL','PART B PROFESSIONAL','PROFESSIONAL','PART A PROFESSIONAL'
]
ALLOWED_V2 = ['P', 'Q']

PREVALIDATION_TASK_KEY = "PreValidation"


def resolve_file_by_prefix(base_dir: str, prefix: str, ext: str = ".gz"):
    try:
        entries = dbutils.fs.ls(base_dir)
    except Exception as e:
        return None, None, None, f"Unable to list dir {base_dir}: {e}"

    rx = re.compile(rf"^({re.escape(prefix)})\.(\d{{6}}|\d{{8}}){re.escape(ext)}$")
    candidates = []
    for it in entries:
        m = rx.match(it.name)
        if m:
            date_digits = m.group(2)
            candidates.append((it.path, it.name, int(date_digits), date_digits, it.modificationTime))

    if not candidates:
        return None, None, None, f"No file for prefix '{prefix}' with 6/8-digit date and {ext}"

    candidates.sort(key=lambda x: (x[2], x[4]), reverse=True)
    best = candidates[0]
    return best[0], best[1], best[3], None

def read_csv_gz(path: str):
    return (spark.read
                 .option("header", "true")
                 .option("delimiter", "|")
                 .csv(path))

def try_get_date_received_from_prevalidation():
    try:
        val = dbutils.jobs.taskValues.get(taskKey=PREVALIDATION_TASK_KEY, key="date_received", debugValue=None)
        return str(val).strip() if val is not None and str(val).strip() else None
    except Exception:
        return None

def fallback_date_received(file1_date_str, file2_date_str):
    d8 = [d for d in [file1_date_str, file2_date_str] if d and len(d) == 8 and d.isdigit()]
    if d8:
        best = max(d8)
        return f"{best[0:4]}-{best[4:6]}-{best[6:8]} 00:00:00"
    return nb_start_str

def summarize_invalids_single_row(df, file_name: str, target_col: str, allowed_values: list,
                                  date_received: str, start_str: str, end_str: str):
    
    col_actual = None
    for c in df.columns:
        if c.lower() == target_col.lower():
            col_actual = c
            break

    if col_actual is None:
        total = df.count()
        return {
            "File_Name": file_name,
            "Date_Received": date_received,
            "Start_Load_Date": start_str,
            "End_Load_Date": end_str,
            "Row_Number": int(total),
            "Error_Description": f"Column '{target_col}' not found; all {total} rows considered invalid."
        }

    invalid_df = df.where(~F.col(col_actual).isin(allowed_values))
    invalid_total = invalid_df.count()
    if invalid_total == 0:
        return None

    val_disp = F.coalesce(F.col(col_actual).cast("string"), F.lit("<NULL>"))
    counts_df = (invalid_df.groupBy(val_disp.alias("value"))
                          .agg(F.count(F.lit(1)).alias("cnt"))
                          .orderBy(F.desc("cnt"), F.asc("value")))
    parts = []
    for r in counts_df.collect():
        v, n = r["value"], int(r["cnt"])
        parts.append(f"Invalid Value = '{v}' (count={n})")

    description = "; ".join(parts)

    return {
        "File_Name": file_name,
        "Date_Received": date_received,
        "Start_Load_Date": start_str,
        "End_Load_Date": end_str,
        "Row_Number": int(invalid_total),
        "Error_Description": description
    }


file1_path, file1_name, file1_date_str, err1 = resolve_file_by_prefix(BASE, FILE1_PREFIX, ".gz")
file2_path, file2_name, file2_date_str, err2 = resolve_file_by_prefix(BASE, FILE2_PREFIX, ".gz")

print(f"File1 -> {file1_name or err1}")
print(f"File2 -> {file2_name or err2}")

DATE_RECEIVED = try_get_date_received_from_prevalidation()
if not DATE_RECEIVED:
    DATE_RECEIVED = fallback_date_received(file1_date_str, file2_date_str)
print(f"Date_Received used for logging: {DATE_RECEIVED}")

nb_end_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

rows_to_write = []

if file1_path is None:
    rows_to_write.append({
        "File_Name": f"{FILE1_PREFIX}.<date>.gz",
        "Date_Received": DATE_RECEIVED,
        "Start_Load_Date": nb_start_str,
        "End_Load_Date": nb_end_str,
        "Row_Number": 0,
        "Error_Description": "File not found by prefix."
    })
else:
    try:
        df1 = read_csv_gz(file1_path)
        r1 = summarize_invalids_single_row(df1, file1_name, COL_NAME, ALLOWED_V1,
                                           DATE_RECEIVED, nb_start_str, nb_end_str)
        if r1: rows_to_write.append(r1)
    except Exception as e:
        rows_to_write.append({
            "File_Name": file1_name,
            "Date_Received": DATE_RECEIVED,
            "Start_Load_Date": nb_start_str,
            "End_Load_Date": nb_end_str,
            "Row_Number": 0,
            "Error_Description": f"Read error: {str(e)}"
        })

if file2_path is None:
    rows_to_write.append({
        "File_Name": f"{FILE2_PREFIX}.<date>.gz",
        "Date_Received": DATE_RECEIVED,
        "Start_Load_Date": nb_start_str,
        "End_Load_Date": nb_end_str,
        "Row_Number": 0,
        "Error_Description": "File not found by prefix."
    })
else:
    try:
        df2 = read_csv_gz(file2_path)
        r2 = summarize_invalids_single_row(df2, file2_name, COL_NAME, ALLOWED_V2,
                                           DATE_RECEIVED, nb_start_str, nb_end_str)
        if r2: rows_to_write.append(r2)
    except Exception as e:
        rows_to_write.append({
            "File_Name": file2_name,
            "Date_Received": DATE_RECEIVED,
            "Start_Load_Date": nb_start_str,
            "End_Load_Date": nb_end_str,
            "Row_Number": 0,
            "Error_Description": f"Read error: {str(e)}"
        })


display(spark.createDataFrame(
    rows_to_write,
    schema=T.StructType([
        T.StructField("File_Name", T.StringType()),
        T.StructField("Date_Received", T.StringType()),
        T.StructField("Start_Load_Date", T.StringType()),
        T.StructField("End_Load_Date", T.StringType()),
        T.StructField("Row_Number", T.LongType()),
        T.StructField("Error_Description", T.StringType()),
    ])
) if rows_to_write else spark.createDataFrame([], schema=T.StructType([
    T.StructField("File_Name", T.StringType()),
    T.StructField("Date_Received", T.StringType()),
    T.StructField("Start_Load_Date", T.StringType()),
    T.StructField("End_Load_Date", T.StringType()),
    T.StructField("Row_Number", T.LongType()),
    T.StructField("Error_Description", T.StringType()),
])))

UPLOAD_TO_TABLE = len(rows_to_write) > 0
ROWS_PREPARED = len(rows_to_write)


dbutils.jobs.taskValues.set(key="error_load_report_flag", value=UPLOAD_TO_TABLE)

print(f"Prepared {ROWS_PREPARED} row(s). UPLOAD_TO_TABLE = {UPLOAD_TO_TABLE}")


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType


Col_Validation_error = False

if UPLOAD_TO_TABLE:
    error_schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received", StringType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True),
        StructField("Row_Number", LongType(), True),
        StructField("Error_Description", StringType(), True),
    ])

    error_df = spark.createDataFrame(rows_to_write, schema=error_schema)

    if error_df.count() > 0:
        Col_Validation_error = True
        error_df.write.format("delta").mode("append").saveAsTable(ERROR_TABLE)
        print(f"❌ Validation errors found. Wrote {error_df.count()} row(s) to {ERROR_TABLE}")
    else:
        print("✅ No validation errors found. Skipping upload.")
else:
    print("⚠️ UPLOAD_TO_TABLE is False. Skipping error upload.")


print(f"Col_Validation_error = {Col_Validation_error}")
dbutils.jobs.taskValues.set(key="Col_Validation_error", value=Col_Validation_error)
